# J-space readout and causal swap

Thin orchestration for the approved spider-to-ant causal minimum.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from jspace_llama.config import ExperimentConfig

In [ ]:
config = ExperimentConfig()
asdict(config)  # Configuration and pinned provenance preview

In [ ]:
from jspace_llama.runtime import (
    download_lens_file,
    load_components,
    require_cuda,
    validate_intervention_tokens,
)

runtime_info = require_cuda()
lens_path = download_lens_file(config.assets)
bundle = load_components(config.assets, lens_path, layers=config.sweep.layers)
validate_intervention_tokens(
    bundle.model.tokenizer, (config.source, config.target)
)

In [ ]:
from jspace_llama.experiment import prepare_prompts
from jspace_llama.readout import read_workspace

prepared_prompts = prepare_prompts(config, bundle.model.tokenizer)
readout = read_workspace(
    bundle.model,
    bundle.dictionary,
    prepared_prompts[0].rendered,
    layers=bundle.layers,
    focus_concept=config.source,
)

In [ ]:
from jspace_llama.experiment import ModelExperimentRunner, run_experiment

runner = ModelExperimentRunner(bundle.model, bundle.dictionary)
experiment = run_experiment(
    config,
    bundle.model.tokenizer,
    layers=bundle.layers,
    generate=runner,
    prepared_prompts=prepared_prompts,
)

In [ ]:
result = {
    "runtime": asdict(runtime_info),
    "readout": readout,
    "experiment": experiment,
}
output_path = Path("../results/readout_and_causal_swap.json")
output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")